# Agent: Response Agent (Evacuation Coordinator)

**Purpose:** Execute evacuation commands and track pedestrian movement.

**Population:** 10 pedestrians

**Locations:**
- Køge Søndre Strand (danger zone): (55.4506, 12.1975)
- Køge Torv (safe zone): (55.4566, 12.1818)

**Behavior:**
- HIGH alert → Evacuate to Køge Torv (8 seconds)
- LOW alert → Return to Køge Søndre Strand (5 seconds)

**Subscribes to:** `city/flood/control/command` topic

In [7]:
import json
import time
import random
from datetime import datetime, timezone

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import ControlCommand, ResponseStatus

cfg = load_config()
mqtt_cfg = cfg.mqtt

# Workshop coordinates
SAFE_LAT, SAFE_LON = 55.456553861769855, 12.181774944848945      # Køge Torv
FLOOD_LAT, FLOOD_LON = 55.45058843620187, 12.197503222443261    # Køge Søndre Strand

NUM_PEDESTRIANS = 10
EVACUATION_TIME_S = 8.0
RETURN_TIME_S = 5.0
UPDATE_INTERVAL_S = 0.5

# Spread pedestrians within 100 meters of the beach
# 100m ≈ 0.0009 degrees, so use ~0.0003 for comfortable spacing
BEACH_SPREAD_LAT = 0.0003
BEACH_SPREAD_LON = 0.0003

# Runtime state
evacuation_mode = False
last_tick = time.time()

people = {
    f"person_{i+1}": {
        "id": f"person_{i+1}",
        "progress": 0.0,
        "location": "strand",
        "speed_factor": 1.3 if i % 7 == 0 else (0.7 if i % 5 == 0 else 1.0),
        "home_lat": FLOOD_LAT + random.uniform(-BEACH_SPREAD_LAT, BEACH_SPREAD_LAT),
        "home_lon": FLOOD_LON + random.uniform(-BEACH_SPREAD_LON, BEACH_SPREAD_LON),
    }
    for i in range(NUM_PEDESTRIANS)
}

print("Response agent configured")
print(f"  Base topic: {mqtt_cfg.base_topic}")
print(f"  Pedestrians: {NUM_PEDESTRIANS}")
print(f"  Evacuation {EVACUATION_TIME_S}s, return {RETURN_TIME_S}s")
print(f"  Pedestrians spread out on beach within ~100m")

Response agent configured
  Base topic: simulated-city
  Pedestrians: 10
  Evacuation 8.0s, return 5.0s
  Pedestrians spread out on beach within ~100m


In [8]:
connector = MqttConnector(mqtt_cfg, client_id_suffix="response")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    raise RuntimeError("Failed to connect to MQTT broker")


def on_control_command(client, userdata, msg):
    global evacuation_mode
    try:
        data = json.loads(msg.payload.decode())
        cmd = ControlCommand.from_json(data)
    except Exception as exc:
        print(f"Control parse error: {exc}")
        return

    if cmd.action != "alert":
        return

    severity = cmd.parameters.get("severity", "low")
    evacuation_mode = severity == "high"
    mode_text = "EVACUATION" if evacuation_mode else "RETURN"
    print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')}] Mode -> {mode_text}")


control_topic = make_topic(mqtt_cfg, "flood", "control", "command")
connector.client.subscribe(control_topic)
connector.client.message_callback_add(control_topic, on_control_command)
print(f"Subscribed to: {control_topic}")

response_topic = make_topic(mqtt_cfg, "flood", "response", "status")
print(f"Publishing status to: {response_topic}")

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


✓ Connected to MQTT broker
Subscribed to: simulated-city/flood/control/command
Publishing status to: simulated-city/flood/response/status


In [9]:
def _interp(start_lat, start_lon, end_lat, end_lon, progress):
    return (
        start_lat + (end_lat - start_lat) * progress,
        start_lon + (end_lon - start_lon) * progress,
    )


def _person_coordinates(person):
    progress = person["progress"]
    home_lat = person["home_lat"]
    home_lon = person["home_lon"]
    
    if evacuation_mode:
        # Evacuate from home position to safe zone
        return _interp(home_lat, home_lon, SAFE_LAT, SAFE_LON, progress)
    if person["location"] == "torv" and progress > 0:
        # Return from safe zone back to home position
        return _interp(SAFE_LAT, SAFE_LON, home_lat, home_lon, progress)
    if person["location"] == "torv":
        # At safe zone, waiting
        return SAFE_LAT, SAFE_LON
    # At home on beach
    return home_lat, home_lon


def update_people(dt):
    for person in people.values():
        duration = EVACUATION_TIME_S if evacuation_mode else RETURN_TIME_S
        speed = person["speed_factor"]
        person["progress"] = min(person["progress"] + (dt * speed / duration), 1.0)

        if person["progress"] >= 1.0:
            person["location"] = "torv" if evacuation_mode else "strand"
            person["progress"] = 0.0


def build_status_payload():
    pedestrians = []
    evacuated = 0

    for person in people.values():
        lat, lon = _person_coordinates(person)
        if person["location"] == "torv":
            evacuated += 1

        pedestrians.append(
            {
                "id": person["id"],
                "lat": lat,
                "lon": lon,
                "location": person["location"],
                "moving": person["progress"] > 0,
                "speed_factor": person["speed_factor"],
            }
        )

    in_transit = sum(1 for p in pedestrians if p["moving"])
    return {
        "evacuated": evacuated,
        "in_transit": in_transit,
        "mode": "evacuation" if evacuation_mode else "return",
        "pedestrians": pedestrians,
    }


def publish_status():
    details = build_status_payload()
    status = ResponseStatus(
        device="response-agent",
        status="running",
        details=details,
        timestamp=datetime.now(timezone.utc).isoformat(),
    )
    payload = json.dumps(status.to_json())
    connector.client.publish(response_topic, payload, qos=1)
    print(
        f"  Evacuated: {details['evacuated']}/{NUM_PEDESTRIANS} | "
        f"In transit: {details['in_transit']}"
    )

In [10]:
print("\nStarting evacuation response loop...")
print(f"Update interval: {UPDATE_INTERVAL_S}s")
print("Press Ctrl+C to stop\n")

try:
    while True:
        now = time.time()
        dt = max(0.0, now - last_tick)
        last_tick = now

        update_people(dt)
        print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')}] Position update")
        publish_status()
        time.sleep(UPDATE_INTERVAL_S)
except KeyboardInterrupt:
    print("\n\n✓ Response agent stopped by user")
finally:
    connector.disconnect()


Starting evacuation response loop...
Update interval: 0.5s
Press Ctrl+C to stop

[08:29:06] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:07] Position update
  Evacuated: 0/10 | In transit: 10


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[08:29:07] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:08] Position update
  Evacuated: 0/10 | In transit: 10


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[08:29:08] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:09] Position update
  Evacuated: 0/10 | In transit: 10


...

[08:29:56] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:56] Position update
  Evacuated: 0/10 | In transit: 3


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[08:29:57] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:57] Position update
  Evacuated: 0/10 | In transit: 10


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[08:29:58] Position update
  Evacuated: 0/10 | In transit: 10
[08:29:58] Position update
  Evacuated: 0/10 | In transit: 8


✓ Response agent stopped by user


In [ ]:
print("This legacy cell is disabled. Use the cells above for the active response loop.")

This legacy cell is disabled. Use the cells above for the active response loop.


## Response Loop

Main loop that:
1. Processes evacuation/return commands
2. Updates pedestrian positions
3. Reports status to dashboard

In [ ]:
print("This legacy callback/publish cell is disabled. Use the active response cells above.")

This legacy callback/publish cell is disabled. Use the active response cells above.


## Evacuation Logic and Callbacks

In [ ]:
print("This legacy connection cell is disabled. Use the active MQTT setup cell above.")

This legacy connection cell is disabled. Use the active MQTT setup cell above.


## Connect to MQTT Broker

In [ ]:
print("This legacy setup cell is disabled. Use the active setup cell at the top of this notebook.")

This legacy setup cell is disabled. Use the active setup cell at the top of this notebook.


## Setup and Configuration

# Agent: Response (Evacuation & Pedprogram)

**Role:** Executes evacuation directives and manages pedestrian movement.

**Responsibilities:**
1. Listen for alert commands from Control agent
2. Track evacuation status of 10 people
3. Simulate movement from Køge Søndre Strand (beach) → Køge Torv (safe zone)
4. Resume normal activity when all-clear issued
5. Publish status updates to dashboard

**People Management:**
- 10 pedestrians initially at Køge Søndre Strand
- On HIGH alert: move to Køge Torv (takes ~8 seconds)
- On LOW alert: return to Køge Søndre Strand (takes ~5 seconds)
- Report location and evacuation status

# Agent Response
Actuator notebook that listens for `ControlCommand` messages and publishes `ResponseStatus` updates.